# scooby step 1: are the OneK1K cell embeddings available?

The OneK1K scooby model needs, besides its weights, one **10-number scPoli vector per cell**.
Its decoder turns each vector into the weights of that cell's output layer.
Hugging Face has only the weights, so the per-cell vectors must come from the two Zenodo zips
(28.5 GB + 49.7 GB). This notebook checks them **without downloading the zips**:

1. reads the header of the model file on Hugging Face, to get the embedding size the decoder expects;
2. reads only the table of contents of each zip and searches it;
3. extracts only the two small files we need;
4. checks that they fit the model.

**Expected:** `embedding_no_val_genes.pq` with 10 numbers per cell for about 1.27 million cells,
and `celltype_fixed.pq`, whose row numbers all point into that table.

File names come from the authors' code (`scooby_reproducibility/scripts/inference_eqtls`).

In [ ]:
%pip install -q remotezip huggingface_hub pandas pyarrow

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

HF_REPO = "lauradmartens/onek1k-scooby"
ZIPS = {
    "resources": "https://zenodo.org/records/15517764/files/reproducibility_data.zip?download=1",
    "training": "https://zenodo.org/records/15517072/files/training_data.zip?download=1",
}
OUT = Path("scooby_assets")
OUT.mkdir(exist_ok=True)

# Filled in automatically in step 3. If the automatic choice is wrong, set them by hand as
# (zip key, path inside the zip), for example ("resources", "reproducibility_data/.../embedding_no_val_genes.pq").
EMB_FILE = None
CELLTYPE_FILE = None

MAX_EXTRACT_GB = 2  # refuse to extract anything bigger by accident
results = {}        # filled in as we go; printed at the end

## 1. What the model expects
Reads only the header of the 765 MB `model.safetensors` (tensor names and shapes).
`cell_state_to_conv.0.weight` has shape (128, *d*), where *d* is the cell-embedding size.

In [ ]:
from huggingface_hub import get_safetensors_metadata

model_dim = None
try:
    meta = get_safetensors_metadata(HF_REPO)
    shapes = {name: info.shape
              for f in meta.files_metadata.values()
              for name, info in f.tensors.items()
              if name.startswith("cell_state_to_conv")}
    for name in sorted(shapes, key=lambda n: (int(n.split(".")[1]), n)):
        print(f"{name:32s} {shapes[name]}")
    model_dim = shapes["cell_state_to_conv.0.weight"][1]
    last = max((n for n in shapes if n.endswith("weight")), key=lambda n: int(n.split(".")[1]))
    n_tracks = shapes[last][0] // 1921  # 1920 filter weights + 1 bias per output track
    print(f"\nDecoder input: {model_dim}-dim cell embedding. Output: {n_tracks} tracks (RNA + and - strand).")
    results["model expects (dims)"] = model_dim
except Exception as e:
    print(f"Could not read the model header: {type(e).__name__}: {e}")
    model_dim = 10  # what the authors pass as cell_emb_dim when loading this model
    results["model expects (dims)"] = "10 (from the authors' code; header not read)"

## 2. List the zips without downloading them
A zip keeps its table of contents at the end of the file. `remotezip` uses HTTP range requests
to fetch only that part (a few MB), not the whole archive.

In [ ]:
import zipfile

from remotezip import RemoteZip


def open_zip(source):
    if Path(source).exists():  # a zip you already downloaded (see the fallback in the last cell)
        return zipfile.ZipFile(source)
    try:
        return RemoteZip(source)
    except Exception as first:
        # Some servers refuse "last N bytes" requests; this variant asks for the file size first.
        try:
            return RemoteZip(source, support_suffix_range=False)
        except Exception as second:
            raise RuntimeError(f"{type(first).__name__}: {str(first)[:200]} | "
                               f"retry: {type(second).__name__}: {str(second)[:200]}") from second


listings = {}
for key, url in ZIPS.items():
    try:
        with open_zip(url) as z:
            df = pd.DataFrame([(i.filename, i.file_size) for i in z.infolist()], columns=["path", "size"])
    except Exception as e:
        print(f"[{key}] could not read the zip remotely: {type(e).__name__}: {e}")
        print("        Use the fallback commands in the last cell instead.")
        continue
    df = df[~df["path"].str.endswith("/")]  # drop folder entries
    df.to_csv(OUT / f"listing_{key}.csv", index=False)
    listings[key] = df
    print(f"[{key}] {len(df):,} files, {df['size'].sum() / 1e9:.1f} GB unpacked; full list: {OUT}/listing_{key}.csv")

In [ ]:
LOOK_FOR = {
    "cell embeddings": r"embedding[^/]*\.(pq|parquet)$",
    "cell-type index": r"cell_?type[^/]*\.(pq|parquet)$",
    "scPoli model (needed later to map CIMA cells)": r"scpoli",
    "fine-mapped eQTL variant tables": r"highpip|susie",
    "authors' precomputed eQTL predictions": r"eqtl_fig4/.*\.h5ad$",
}


def search(pattern):
    rows = [(key, path, size)
            for key, df in listings.items()
            for path, size in zip(df["path"], df["size"])
            if re.search(pattern, path, re.IGNORECASE)]
    return pd.DataFrame(rows, columns=["zip", "path", "size"])


for label, pattern in LOOK_FOR.items():
    hits = search(pattern)
    print(f"\n== {label}: {len(hits)} file(s)")
    for _, r in hits.head(15).iterrows():
        print(f"   {r['size'] / 1e6:9.1f} MB  [{r['zip']}] {r['path']}")
    if len(hits) > 15:
        print(f"   ... and {len(hits) - 15} more (see the listing CSVs)")
    results[f"files found: {label}"] = len(hits)

## 3. Extract only the OneK1K files
Picks paths that contain `onek1k`, preferring the exact file names the authors' code loads.
The NeurIPS bone-marrow model has its own files (14-dim), which must not be used with this model.

In [ ]:
def pick(pattern, file_name):
    hits = search(pattern)
    hits = hits[hits["path"].str.contains("onek1k", case=False)]
    if hits.empty:
        return None
    exact = hits["path"].str.rsplit("/", n=1).str[-1] == file_name
    hits = hits.assign(exact=exact).sort_values("exact", ascending=False)
    return tuple(hits.iloc[0][["zip", "path"]])


def extract(item):
    key, path = item
    with open_zip(ZIPS[key]) as z:
        size = z.getinfo(path).file_size
        if size > MAX_EXTRACT_GB * 1e9:
            raise ValueError(f"{path} is {size / 1e9:.1f} GB; raise MAX_EXTRACT_GB if you really want it")
        return Path(z.extract(path, OUT))


EMB_FILE = EMB_FILE or pick(LOOK_FOR["cell embeddings"], "embedding_no_val_genes.pq")
CELLTYPE_FILE = CELLTYPE_FILE or pick(LOOK_FOR["cell-type index"], "celltype_fixed.pq")
print("embeddings:", EMB_FILE)
print("cell types:", CELLTYPE_FILE)

emb_path = extract(EMB_FILE) if EMB_FILE else None
ct_path = extract(CELLTYPE_FILE) if CELLTYPE_FILE else None
print("saved to:", emb_path, "|", ct_path)

## 4. Check the embeddings
Expect one row per cell and 10 numbers per row, matching the model in step 1.

In [ ]:
emb = None
if emb_path:
    emb = pd.read_parquet(emb_path)
    X = np.vstack(emb["embedding"].to_numpy())
    print("columns:", list(emb.columns))
    print(f"cells: {X.shape[0]:,} | numbers per cell: {X.shape[1]} | dtype: {X.dtype} | "
          f"non-finite values: {int((~np.isfinite(X)).sum())}")
    print(emb.head(3), "\n")
    print("per-dimension summary:")
    print(pd.DataFrame(X).describe().T[["mean", "std", "min", "max"]].round(2))
    results["embedding: cells"] = X.shape[0]
    results["embedding: dims"] = X.shape[1]
    results["embedding dims match the model"] = X.shape[1] == model_dim
else:
    print("No OneK1K embedding file found. See the last cell for what to do.")

## 5. Check the cell-type index
Each row is a cell type plus the row numbers of its cells in the embedding table.
Predictions for a cell type are summed over exactly these cells.

In [ ]:
if ct_path:
    ct = pd.read_parquet(ct_path)
    ct["n_cells"] = ct["cellindex"].apply(len)
    print(ct[["celltype", "n_cells"]].sort_values("n_cells", ascending=False).to_string(index=False))
    idx = np.concatenate(ct["cellindex"].to_numpy()).astype(np.int64)
    n_twice = len(idx) - len(np.unique(idx))
    print(f"\n{len(ct)} cell types, {len(idx):,} cells assigned, {n_twice:,} assigned more than once")
    results["cell types"] = len(ct)
    if emb is not None and len(idx):
        inside = bool(idx.min() >= 0 and idx.max() < len(emb))
        print(f"row numbers run {idx.min():,}..{idx.max():,}; embedding has {len(emb):,} rows ->",
              "OK" if inside else "MISMATCH")
        results["cell-type rows point inside the embedding"] = inside
else:
    print("No OneK1K cell-type index found.")

In [ ]:
print(pd.Series(results, dtype=object).to_string())

## What the result means

- **Embedding found, 10 dims, about 1.27M cells, and the cell-type rows fit:** you have everything
  the decoder needs. Go to step 2: build the environment on the Blackwell GPU and run a smoke test.
  If the authors' precomputed eQTL predictions (`eQTL_fig4/…h5ad`) were also found, keep them:
  your own run should reproduce those numbers, which is the best test of your setup.
- **No embedding, but an scPoli model was found:** the exact embedding can be recomputed from it
  (`scPoli.load(...)` then `get_latent(adata, mean=True)` on the same 2,403 genes the authors used).
- **Neither found:** open an issue on github.com/gagneurlab/scooby asking for the OneK1K
  `embedding_no_val_genes.pq`, `celltype_fixed.pq` and the scPoli model. Do not retrain scPoli
  yourself: a new run gives a different latent space, and the decoder would produce meaningless output.

**Fallback if step 2 cannot read the zips remotely:** download the zip in a terminal
(`-c` resumes a broken download), then put the local file name in `ZIPS` and run the notebook again.
```bash
wget -c -O reproducibility_data.zip "https://zenodo.org/records/15517764/files/reproducibility_data.zip?download=1"
```
```python
ZIPS["resources"] = "reproducibility_data.zip"
```